# 04 - Explainable AIm

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


In [4]:
from src.config.settings import get_settings
from src.data.loader import SessionDataLoader
from src.features.engineering import add_engineered_features
from src.preprocessing.pipeline import get_feature_columns
from src.utils.io import load_artifact, load_json

settings = get_settings()
explainer = load_artifact(settings.paths.resolve('explainer_artifact'))
report = load_json(settings.paths.resolve('metrics_report'))
df = add_engineered_features(SessionDataLoader(settings=settings).load())
feature_cols = get_feature_columns(settings)
print('Best model:', report['best_model'])

FileNotFoundError: Artifact not found at C:\Desktop\Inernships\Projects\E-Commerce Purchase Intent Prediction using Explainable AI\models\shap_explainer.joblib

## Global SHAP summary

In [ ]:
global_result = explainer.global_shap_values(df[feature_cols], max_rows=800)
top15 = global_result['feature_importance'][:15]
imp_df = pd.DataFrame(top15)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df['feature'][::-1], imp_df['mean_abs_shap'][::-1], color='#3b6ea5')
ax.set_xlabel('mean |SHAP value|')
ax.set_title('Global SHAP feature importance')
plt.tight_layout()
plt.show()

**Takeaway:** PageValues dominates the global ranking, exactly as flagged in the EDA and the PRD's watch-out. This is the prompt to revisit the leakage question: is PageValues actually known at the moment you'd score a session in production? See the PageValues ablation in `docs/model_card.md` / `models/metrics_report.json` for the documented decision and the honest performance gap without it.

## Local explaination - a single session (SHAP)

In [ ]:
sample_row = df[feature_cols].iloc[[42]]
local_shap = explainer.local_shap_explaination(sample_row, top_k=8)
pd.DataFrame(local_shap['top_contributors'])

## Same session - independent cross-check (LIME)

In [ ]:
local_lime = explainer.local_lime_explanation(sample_row, top_k=8)
pd.DataFrame(local_lime['top_contributors'])

**Takeaway:** SHAP and LIME agree on the top driver (PageValues) for this session, and broadly agree on direction for the next few features — two independent explanation methods pointing the same way is good evidence the explanation reflects the model's real behaviour, not an artefact of one method's approximation.

Both explanations are also surfaced live in the FastAPI `/predict` response and the dashboard's Live Session Scoring / Explainability sections — not just here in the notebook, per the PRD's Done-when criteria.